# 02 · Emparejar los partidos de D1 (FotMob) y D2 (Kaggle)

Cada dataset usa su propio ID de partido. Aquí se construye la correspondencia entre ambos, emparejando por las dos selecciones y la fecha.

**Lee:** `data/d1/matches/*.json` (solo la sección `general`) y `data/d2/matches_detailed.csv`
**Genera:** `data/processed/mapa_d1_d2.csv`

In [1]:
import json
from pathlib import Path
import pandas as pd

D1 = Path("../data/d1/matches")
D2 = Path("../data/d2")
PROCESADOS = Path("../data/processed")
PROCESADOS.mkdir(parents=True, exist_ok=True)

def info_partido(ruta):
    d = json.load(open(ruta, encoding="utf-8"))
    g = d["general"]
    # matchName viene como "Germany-vs-Paraguay_Mon, Jun 29, 2026, 20:30 UTC"
    texto_fecha = g["matchName"].split("_", 1)[1].replace(" UTC", "")
    fecha = pd.to_datetime(texto_fecha, format="%a, %b %d, %Y, %H:%M")
    return {"fotmob_id": int(g["matchId"]), "fecha": str(fecha.date()),
            "home": g["homeTeam"]["name"], "away": g["awayTeam"]["name"]}

d1 = pd.DataFrame([info_partido(f) for f in sorted(D1.glob("*.json"))])
d2 = pd.read_csv(D2 / "matches_detailed.csv")[["match_id", "date", "home_team_name", "away_team_name"]]
print("partidos en D1:", len(d1), "| partidos en D2:", len(d2))

partidos en D1: 104 | partidos en D2: 104


## 1. Unificar los nombres de las selecciones

Seis selecciones están escritas distinto en cada dataset. Se usan los nombres de D2 como referencia.

In [2]:
nombres_d1 = set(d1.home) | set(d1.away)
nombres_d2 = set(d2.home_team_name) | set(d2.away_team_name)
print("Solo en D1:", sorted(nombres_d1 - nombres_d2))
print("Solo en D2:", sorted(nombres_d2 - nombres_d1))

Solo en D1: ['Cape Verde', 'Curacao', 'DR Congo', 'Iran', 'Ivory Coast', 'Turkiye']
Solo en D2: ['Cabo Verde', 'Congo DR', 'Curaçao', "Côte d'Ivoire", 'IR Iran', 'Türkiye']


In [3]:
equiv = {"Cape Verde": "Cabo Verde", "Curacao": "Curaçao", "DR Congo": "Congo DR",
         "Iran": "IR Iran", "Ivory Coast": "Côte d'Ivoire", "Turkiye": "Türkiye"}
d1["home"] = d1["home"].replace(equiv)
d1["away"] = d1["away"].replace(equiv)

# después de unificar, no debe quedar ningún nombre sin pareja
nombres_d1 = set(d1.home) | set(d1.away)
print("Nombres sin pareja:", sorted(nombres_d1 ^ nombres_d2))

Nombres sin pareja: []


## 2. Emparejar por selecciones y fecha

Las fechas difieren en ±1 día en parte de los partidos (en ambas direcciones), así que se empareja por el par de selecciones, ordenado alfabéticamente, con una tolerancia de un día.

In [4]:
d1["par"] = [tuple(sorted(p)) for p in zip(d1.home, d1.away)]
d2["par"] = [tuple(sorted(p)) for p in zip(d2.home_team_name, d2.away_team_name)]

cand = d1.merge(d2, on="par")      # todas las combinaciones con el mismo par de selecciones
cand["dif_dias"] = (pd.to_datetime(cand.fecha) - pd.to_datetime(cand.date)).dt.days
print("Diferencia de fecha entre D1 y D2 (días):")
print(cand.dif_dias.value_counts())

union = cand[cand.dif_dias.abs() <= 1].copy()
union["invertido"] = union.home != union.home_team_name
print("filas:", len(union),
      "| partidos D1:", union.fotmob_id.nunique(),
      "| partidos D2:", union.match_id.nunique(),
      "| local invertido:", union.invertido.sum())

# la correspondencia debe ser 1 a 1 y cubrir los 104 partidos
assert len(union) == union.fotmob_id.nunique() == union.match_id.nunique() == 104

Diferencia de fecha entre D1 y D2 (días):
dif_dias
 0    62
 1    24
-1    18
Name: count, dtype: int64
filas: 104 | partidos D1: 104 | partidos D2: 104 | local invertido: 0


In [5]:
union[["fotmob_id", "match_id", "invertido"]].to_csv(PROCESADOS / "mapa_d1_d2.csv", index=False)
print("Guardado:", PROCESADOS / "mapa_d1_d2.csv")

Guardado: ../data/processed/mapa_d1_d2.csv
